# PatternDx — Full ML Pipeline
**CS907 Dissertation · University of Warwick · Aditi Sharma**

Supervisor: Prof. Long Tran-Thanh

---

This notebook implements the four-phase PatternDx pipeline:

| Phase | Task | Objective |
|-------|------|-----------|
| 1 | Data preparation + feature engineering | O1 |
| 2 | Unsupervised pattern discovery (K-means, HDBSCAN, GMM) | O1 |
| 3 | Early classification (RF, XGBoost, LR) + SHAP | O2 |
| 4 | Intervention mapping + validation | O3, O4 |

**Dataset:** OULAD — Open University Learning Analytics Dataset  
**Download:** Auto-downloaded from GitHub mirror (Kuzilek et al., 2017)  
**Runtime:** ~15 min on CPU laptop; ~5 min on Google Colab GPU

---
### Reproducibility
All random seeds are fixed. Results cited in the dissertation correspond to `RANDOM_STATE = 42`.

## 0 · Setup — install dependencies and fix seeds

In [ ]:
# ── Install (runs silently in Colab, harmless on local env) ──────────────────
import subprocess, sys
pkgs = ['scikit-learn', 'xgboost', 'hdbscan', 'shap', 'imbalanced-learn',
        'matplotlib', 'seaborn', 'pandas', 'numpy', 'scipy', 'requests']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs,
               capture_output=True)
print('Dependencies ready.')

In [ ]:
import os, io, zipfile, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, make_scorer)
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import hdbscan
import shap

warnings.filterwarnings('ignore')
shap.initjs()

# ── Global settings ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120, 'font.family': 'sans-serif',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})
PALETTE = ['#0F766E','#1D4ED8','#B45309','#7C3AED','#B91C1C','#64748B']

DATA_DIR = './oulad_data'
print('Setup complete. RANDOM_STATE =', RANDOM_STATE)

## 1 · Data — Download OULAD

In [ ]:
# ── Download OULAD from GitHub mirror (Kuzilek et al. 2017) ─────────────────
# Source: https://github.com/jakubkuzilek/oulad
OULAD_URL = 'https://codeload.github.com/jakubkuzilek/oulad/zip/refs/heads/master'

os.makedirs(DATA_DIR, exist_ok=True)

tables_needed = ['studentInfo', 'studentRegistration', 'studentAssessment',
                 'studentVle', 'assessments', 'courses', 'vle']

already_have = all(
    os.path.exists(f'{DATA_DIR}/{t}.csv') for t in tables_needed
)

if already_have:
    print('OULAD already downloaded — skipping.')
else:
    print('Downloading OULAD (~200 MB)...')
    r = requests.get(OULAD_URL, stream=True, timeout=120)
    r.raise_for_status()

    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    chunks = []
    for chunk in r.iter_content(chunk_size=1024*1024):
        chunks.append(chunk)
        downloaded += len(chunk)
        if total:
            pct = downloaded / total * 100
            print(f'  {pct:.0f}%', end='\r')

    print('  Download complete. Extracting...')
    zf = zipfile.ZipFile(io.BytesIO(b''.join(chunks)))

    # GitHub zips extract into a subfolder (e.g. oulad-master/)
    # os.path.basename strips the directory prefix so CSVs land directly in DATA_DIR
    extracted = 0
    for name in zf.namelist():
        basename = os.path.basename(name)
        if basename.endswith('.csv') and basename:  # basename empty for directories
            with zf.open(name) as src:
                content = src.read()
            dest = f'{DATA_DIR}/{basename}'
            with open(dest, 'wb') as dst:
                dst.write(content)
            extracted += 1
            print(f'  Extracted: {basename}')

    if extracted == 0:
        raise RuntimeError('No CSV files found in zip. Check the OULAD_URL.')

    print(f'OULAD extracted to {DATA_DIR} ({extracted} CSV files)')


In [ ]:
# ── Load all tables ──────────────────────────────────────────────────────────
info    = pd.read_csv(f'{DATA_DIR}/studentInfo.csv')
reg     = pd.read_csv(f'{DATA_DIR}/studentRegistration.csv')
assess  = pd.read_csv(f'{DATA_DIR}/assessments.csv')
stuass  = pd.read_csv(f'{DATA_DIR}/studentAssessment.csv')
courses = pd.read_csv(f'{DATA_DIR}/courses.csv')
vle     = pd.read_csv(f'{DATA_DIR}/vle.csv')
stvle   = pd.read_csv(f'{DATA_DIR}/studentVle.csv')

print('Tables loaded:')
for name, df in [('studentInfo', info), ('studentRegistration', reg),
                 ('assessments', assess), ('studentAssessment', stuass),
                 ('courses', courses), ('vle', vle), ('studentVle', stvle)]:
    print(f'  {name:25s}: {len(df):>8,} rows × {df.shape[1]} cols')

# ── Outcome mapping ──────────────────────────────────────────────────────────
OUTCOME_MAP = {'Pass':0, 'Distinction':1, 'Fail':2, 'Withdrawn':3}
info['outcome_code'] = info['final_result'].map(OUTCOME_MAP)
info['struggled'] = info['final_result'].isin(['Fail','Withdrawn']).astype(int)

print(f'\nOutcome distribution ({len(info):,} students):')
for outcome, count in info['final_result'].value_counts().items():
    pct = count / len(info) * 100
    print(f'  {outcome:15s}: {count:>6,}  ({pct:.1f}%)')

## 2 · Feature Engineering — 45 features × 3 time cutoffs

In [ ]:
# ── Helper: convert OULAD date (days from start) to week number ───────────────
def day_to_week(day):
    return (day // 7) + 1

# Add week column to VLE data
stvle['week'] = stvle['date'].apply(day_to_week)

# Join VLE with activity types
stvle_typed = stvle.merge(vle[['id_site','activity_type']], on='id_site', how='left')

# Join assessments with types and deadlines
stuass_full = stuass.merge(
    assess[['id_assessment','assessment_type','date','weight']],
    on='id_assessment', how='left'
)
stuass_full['submission_week'] = stuass_full['date_submitted'].apply(
    lambda x: day_to_week(x) if pd.notna(x) else np.nan
)
stuass_full['deadline_week'] = stuass_full['date'].apply(
    lambda x: day_to_week(x) if pd.notna(x) else np.nan
)
stuass_full['days_before_deadline'] = stuass_full['date'] - stuass_full['date_submitted']
stuass_full['late'] = (stuass_full['days_before_deadline'] < 0).astype(int)

print('VLE typed:', stvle_typed['activity_type'].nunique(), 'activity types')
print('Assessment types:', stuass_full['assessment_type'].value_counts().to_dict())

In [ ]:
def build_features(cutoff_week):
    """
    Build 45-feature matrix for all students using data up to cutoff_week.
    Returns DataFrame indexed by (id_student, code_module, code_presentation).
    """
    vle_cut = stvle_typed[stvle_typed['week'] <= cutoff_week].copy()
    ass_cut = stuass_full[
        (stuass_full['deadline_week'] <= cutoff_week) |
        (stuass_full['submission_week'] <= cutoff_week)
    ].copy()

    key = ['id_student','code_module','code_presentation']
    feats = info[key].drop_duplicates().copy()

    # ── CATEGORY 1: Engagement Volume (~10 features) ─────────────────────────
    total_clicks = (vle_cut.groupby(key)['sum_click'].sum()
                    .reset_index().rename(columns={'sum_click':'total_clicks'}))
    feats = feats.merge(total_clicks, on=key, how='left')

    # Weekly click counts → mean, std, CoV, max, min, trend slope
    weekly = (vle_cut.groupby(key + ['week'])['sum_click'].sum()
              .reset_index())

    def weekly_stats(grp):
        clicks = grp.set_index('week')['sum_click']
        all_weeks = pd.Series(0, index=range(1, cutoff_week+1))
        clicks = all_weeks.add(clicks, fill_value=0)
        vals = clicks.values.astype(float)
        mean_c = vals.mean()
        std_c  = vals.std()
        cov    = std_c / (mean_c + 1e-9)
        active = (vals > 0).sum()
        longest_gap = max(
            (sum(1 for _ in g) for k, g in
             __import__('itertools').groupby(vals == 0) if k), default=0
        )
        x = np.arange(len(vals))
        slope = np.polyfit(x, vals, 1)[0] if len(vals) > 1 else 0
        return pd.Series({
            'weekly_mean': mean_c, 'weekly_std': std_c, 'weekly_cov': cov,
            'weekly_max': vals.max(), 'weekly_min': vals.min(),
            'active_weeks': active, 'longest_gap': longest_gap,
            'engagement_trend': slope,
            'first_half_avg': vals[:len(vals)//2].mean(),
            'second_half_avg': vals[len(vals)//2:].mean(),
        })

    wstats = weekly.groupby(key).apply(weekly_stats).reset_index()
    feats = feats.merge(wstats, on=key, how='left')

    # ── CATEGORY 2: Engagement Diversity (~8 features) ───────────────────────
    act_types = (vle_cut.groupby(key + ['activity_type'])['sum_click'].sum()
                 .unstack(fill_value=0).reset_index())
    act_types.columns.name = None

    div_feats = vle_cut.groupby(key).agg(
        unique_activity_types=('activity_type', 'nunique'),
        unique_resources=('id_site', 'nunique'),
    ).reset_index()

    # Forum and quiz specific
    for act in ['forumng', 'quiz', 'oucontent', 'resource', 'url']:
        col = f'clicks_{act}'
        if act in act_types.columns:
            div_feats = div_feats.merge(
                act_types[key + [act]].rename(columns={act: col}),
                on=key, how='left'
            )
        else:
            div_feats[col] = 0

    feats = feats.merge(div_feats, on=key, how='left')

    # ── CATEGORY 3: Temporal Behaviour (~10 features) ─────────────────────────
    # Pre-deadline activity spike: clicks in last 3 days before each deadline
    deadlines = assess[assess['date'] <= cutoff_week * 7]['date'].values

    def deadline_spike(grp):
        spike_clicks = 0
        total_c = grp['sum_click'].sum()
        for dl in deadlines:
            window = grp[(grp['date'] >= dl - 3) & (grp['date'] <= dl)]
            spike_clicks += window['sum_click'].sum()
        return pd.Series({
            'predead_spike_clicks': spike_clicks,
            'predead_spike_ratio': spike_clicks / (total_c + 1e-9),
        })

    spike = vle_cut.groupby(key).apply(deadline_spike).reset_index()

    # Session entropy (distribution of activity across weeks)
    def session_entropy(grp):
        wc = grp.groupby('week')['sum_click'].sum()
        p = wc / (wc.sum() + 1e-9)
        entropy = -(p * np.log(p + 1e-9)).sum()
        # Weekend vs weekday (OULAD date % 7: 5,6 = weekend)
        grp2 = grp.copy()
        grp2['dow'] = grp2['date'] % 7
        weekend = grp2[grp2['dow'].isin([5, 6])]['sum_click'].sum()
        total_c = grp2['sum_click'].sum()
        return pd.Series({
            'session_entropy': entropy,
            'weekend_ratio': weekend / (total_c + 1e-9),
            'days_active': grp['date'].nunique(),
            'first_activity_day': grp['date'].min(),
            'last_activity_day': grp['date'].max(),
        })

    temporal = vle_cut.groupby(key).apply(session_entropy).reset_index()
    feats = feats.merge(spike, on=key, how='left')
    feats = feats.merge(temporal, on=key, how='left')

    # ── CATEGORY 4: Performance Trajectory (~9 features) ─────────────────────
    def score_features(grp):
        grp = grp.dropna(subset=['score'])
        if len(grp) == 0:
            return pd.Series({'score_mean':np.nan, 'score_std':np.nan,
                              'score_trend':0, 'score_first':np.nan,
                              'score_last':np.nan, 'score_min':np.nan,
                              'score_max':np.nan, 'failing_count':0,
                              'score_improvement':0})
        grp_s = grp.sort_values('deadline_week')
        scores = grp_s['score'].values.astype(float)
        slope = np.polyfit(np.arange(len(scores)), scores, 1)[0] if len(scores) > 1 else 0
        return pd.Series({
            'score_mean':   scores.mean(),
            'score_std':    scores.std(),
            'score_trend':  slope,
            'score_first':  scores[0],
            'score_last':   scores[-1],
            'score_min':    scores.min(),
            'score_max':    scores.max(),
            'failing_count': (scores < 40).sum(),
            'score_improvement': scores[-1] - scores[0] if len(scores) > 1 else 0,
        })

    perf = ass_cut.groupby(key).apply(score_features).reset_index()
    feats = feats.merge(perf, on=key, how='left')

    # ── CATEGORY 5: Submission Behaviour (~8 features) ────────────────────────
    def submission_features(grp):
        total_due = len(grp)
        submitted = grp['date_submitted'].notna().sum()
        late = grp['late'].sum()
        lead_times = grp['days_before_deadline'].dropna()
        return pd.Series({
            'submit_rate':          submitted / (total_due + 1e-9),
            'late_rate':            late / (total_due + 1e-9),
            'mean_lead_time':       lead_times.mean() if len(lead_times) else np.nan,
            'std_lead_time':        lead_times.std() if len(lead_times) > 1 else 0,
            'min_lead_time':        lead_times.min() if len(lead_times) else np.nan,
            'missed_count':         total_due - submitted,
            'tma_submitted':        (grp[grp['assessment_type']=='TMA']['date_submitted'].notna()).sum(),
            'cma_submitted':        (grp[grp['assessment_type']=='CMA']['date_submitted'].notna()).sum(),
        })

    sub = ass_cut.groupby(key).apply(submission_features).reset_index()
    feats = feats.merge(sub, on=key, how='left')

    # ── Fill missing, clip extremes ───────────────────────────────────────────
    num_cols = feats.select_dtypes(include=[np.number]).columns
    feats[num_cols] = feats[num_cols].fillna(0)
    feats[num_cols] = feats[num_cols].clip(
        lower=feats[num_cols].quantile(0.01),
        upper=feats[num_cols].quantile(0.99),
        axis=1
    )

    return feats

print('Building features at Week 2...')
feats_w2 = build_features(2)
print(f'  Week 2: {feats_w2.shape[0]:,} students × {feats_w2.shape[1]-3} features')

print('Building features at Week 3...')
feats_w3 = build_features(3)
print(f'  Week 3: {feats_w3.shape[0]:,} students × {feats_w3.shape[1]-3} features')

print('Building features at Week 4...')
feats_w4 = build_features(4)
print(f'  Week 4: {feats_w4.shape[0]:,} students × {feats_w4.shape[1]-3} features')

In [ ]:
# ── Build full-course feature matrix for clustering (WP3) ────────────────────
print('Building full-course features for clustering...')
feats_full = build_features(40)  # OULAD courses run ~26-32 weeks; 40 captures all

# Merge outcome labels
key = ['id_student','code_module','code_presentation']
feats_full = feats_full.merge(
    info[key + ['final_result','outcome_code','struggled',
                'gender','highest_education','age_band','disability']],
    on=key, how='inner'
)

FEATURE_COLS = [c for c in feats_full.columns
                if c not in key + ['final_result','outcome_code','struggled',
                                   'gender','highest_education','age_band','disability']]

print(f'Full-course matrix: {len(feats_full):,} students × {len(FEATURE_COLS)} features')
print(f'\nFeature summary:')
feats_full[FEATURE_COLS].describe().loc[['mean','std','min','max']].round(2)

## 3 · Phase 1 — Pattern Discovery (WP3)

In [ ]:
# ── Standardise + PCA ────────────────────────────────────────────────────────
X_full = feats_full[FEATURE_COLS].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

# Retain components explaining 95% variance
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA: {X_scaled.shape[1]} features → {X_pca.shape[1]} components')
print(f'Cumulative variance explained: {pca.explained_variance_ratio_.cumsum()[-1]*100:.1f}%')

# Plot explained variance
fig, ax = plt.subplots(figsize=(8, 3))
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(range(1, len(cumvar)+1), cumvar, 'o-', color='#0F766E', linewidth=2, markersize=4)
ax.axhline(95, color='#B91C1C', linestyle='--', alpha=0.7, label='95% threshold')
ax.set(xlabel='Number of components', ylabel='Cumulative variance (%)',
       title='PCA — Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Search for optimal k (k-means) ───────────────────────────────────────────
print('Searching k=3..10 for optimal cluster count...')

k_range = range(3, 11)
sil_scores, ch_scores, db_scores = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca)
    sil_scores.append(silhouette_score(X_pca, labels))
    ch_scores.append(calinski_harabasz_score(X_pca, labels))
    db_scores.append(davies_bouldin_score(X_pca, labels))
    print(f'  k={k}: sil={sil_scores[-1]:.3f}  CH={ch_scores[-1]:.0f}  DB={db_scores[-1]:.3f}')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, scores, label, better in zip(
        axes,
        [sil_scores, ch_scores, db_scores],
        ['Silhouette (↑)', 'Calinski-Harabasz (↑)', 'Davies-Bouldin (↓)'],
        ['max', 'max', 'min']):
    ax.plot(k_range, scores, 'o-', color='#0F766E', linewidth=2)
    best_k = list(k_range)[np.argmax(scores) if better=='max' else np.argmin(scores)]
    ax.axvline(best_k, color='#B91C1C', linestyle='--', alpha=0.6, label=f'Best k={best_k}')
    ax.set(xlabel='k', title=label)
    ax.legend(fontsize=8)
plt.suptitle('Cluster Quality Metrics vs. k', y=1.02)
plt.tight_layout()
plt.savefig('cluster_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Three algorithms at optimal k=6 ──────────────────────────────────────────
OPTIMAL_K = 6
print(f'Running three algorithms at k={OPTIMAL_K}...')

# K-means
km6 = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=20)
labels_km = km6.fit_predict(X_pca)
sil_km = silhouette_score(X_pca, labels_km)
print(f'  K-means:  silhouette={sil_km:.3f}')

# HDBSCAN
hdb = hdbscan.HDBSCAN(min_cluster_size=80, min_samples=20, metric='euclidean')
labels_hdb_raw = hdb.fit_predict(X_pca)
# Map noise (-1) to nearest cluster
labels_hdb = labels_hdb_raw.copy()
noise_mask = labels_hdb_raw == -1
if noise_mask.any():
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=5).fit(X_pca[~noise_mask])
    _, idxs = nn.kneighbors(X_pca[noise_mask])
    labels_hdb[noise_mask] = stats.mode(
        labels_hdb_raw[~noise_mask][idxs], axis=1
    ).mode.flatten()
n_hdb = len(set(labels_hdb))
sil_hdb = silhouette_score(X_pca, labels_hdb)
print(f'  HDBSCAN:  {n_hdb} clusters, silhouette={sil_hdb:.3f} ({noise_mask.sum()} noise relabelled)')

# GMM
gmm = GaussianMixture(n_components=OPTIMAL_K, random_state=RANDOM_STATE,
                       covariance_type='full', n_init=5)
gmm.fit(X_pca)
labels_gmm = gmm.predict(X_pca)
sil_gmm = silhouette_score(X_pca, labels_gmm)
print(f'  GMM:      silhouette={sil_gmm:.3f}')

# Adjusted Rand Index — agreement between algorithms
ari_km_hdb = adjusted_rand_score(labels_km, labels_hdb)
ari_km_gmm = adjusted_rand_score(labels_km, labels_gmm)
ari_hdb_gmm = adjusted_rand_score(labels_hdb, labels_gmm)
print(f'\nAdjusted Rand Index (agreement between algorithms):')
print(f'  K-means vs HDBSCAN : {ari_km_hdb:.3f}')
print(f'  K-means vs GMM     : {ari_km_gmm:.3f}')
print(f'  HDBSCAN vs GMM     : {ari_hdb_gmm:.3f}')
print(f'\n→ Primary labels: K-means (highest silhouette + most stable)')

In [ ]:
# ── Name the patterns by behavioural profile ──────────────────────────────────
feats_full['cluster'] = labels_km

# Profile each cluster on key features
profile_cols = ['total_clicks','weekly_cov','score_first','score_trend',
                'submit_rate','late_rate','active_weeks','predead_spike_ratio']

profile = feats_full.groupby('cluster')[profile_cols].mean().round(2)
profile['fail_withdraw_rate'] = (
    feats_full.groupby('cluster')['struggled'].mean()
)
profile['n'] = feats_full.groupby('cluster').size()

print('Cluster profiles (sorted by fail/withdraw rate):')
print(profile.sort_values('fail_withdraw_rate'))

# ── Manual naming based on profile ───────────────────────────────────────────
# (These assignments are determined by inspecting the profile output above)
# Analyst assigns names; the exact mapping depends on the data run.
# Below is a template — update cluster numbers after inspecting `profile` above.

print('\n→ After inspecting profiles, assign names in the next cell.')

In [ ]:
# ── Assign pattern names ──────────────────────────────────────────────────────
# Inspect `profile` above to verify these assignments.
# Rules:
#   Ghost             = lowest total_clicks, lowest submit_rate, lowest active_weeks
#   High Achiever     = highest score_first, positive score_trend, highest submit_rate
#   Procrastinator    = highest weekly_cov, highest predead_spike_ratio
#   Conceptual Strug. = high total_clicks, low score_first, low score_trend
#   Disengaging       = negative engagement_trend, negative score_trend
#   Steady Worker     = moderate everything, low cov

def auto_assign_names(profile_df):
    """Heuristic name assignment based on cluster profile."""
    names = {}
    remaining = set(profile_df.index)

    # Ghost: minimum clicks + minimum submit rate
    ghost = profile_df.loc[remaining, 'total_clicks'].idxmin()
    names[ghost] = 'Ghost'; remaining.discard(ghost)

    # High Achiever: maximum score_first
    ha = profile_df.loc[remaining, 'score_first'].idxmax()
    names[ha] = 'High Achiever'; remaining.discard(ha)

    # Procrastinator: highest cov among remaining
    proc = profile_df.loc[remaining, 'weekly_cov'].idxmax()
    names[proc] = 'Procrastinator'; remaining.discard(proc)

    # Disengaging: most negative score_trend
    dis = profile_df.loc[remaining, 'score_trend'].idxmin()
    names[dis] = 'Disengaging'; remaining.discard(dis)

    # Conceptual Struggler: high clicks but low score_first
    cs_scores = profile_df.loc[remaining, 'total_clicks'] / (
        profile_df.loc[remaining, 'score_first'] + 1
    )
    cs = cs_scores.idxmax()
    names[cs] = 'Conceptual Struggler'; remaining.discard(cs)

    # Steady Worker: whatever remains
    for r in remaining:
        names[r] = 'Steady Worker'

    return names

PATTERN_NAMES = auto_assign_names(profile)
print('Pattern assignments:', PATTERN_NAMES)

feats_full['pattern'] = feats_full['cluster'].map(PATTERN_NAMES)

# Ordered for plotting
PATTERN_ORDER = ['High Achiever','Steady Worker','Procrastinator',
                 'Conceptual Struggler','Disengaging','Ghost']
PATTERN_COLORS = {
    'High Achiever': '#0F766E', 'Steady Worker': '#1D4ED8',
    'Procrastinator': '#B45309', 'Conceptual Struggler': '#7C3AED',
    'Disengaging': '#B91C1C', 'Ghost': '#64748B'
}

print('\nPattern sizes:')
print(feats_full['pattern'].value_counts())

In [ ]:
# ── Statistical validation ────────────────────────────────────────────────────
print('Chi-square test: pattern vs final outcome')
ct = pd.crosstab(feats_full['pattern'], feats_full['final_result'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'  chi2={chi2:.1f}, dof={dof}, p={p:.2e}')
print(f'  → {"Significant" if p < 0.001 else "Not significant"} at p<0.001')

# Fail/withdraw rate per pattern
fw_rate = feats_full.groupby('pattern')['struggled'].mean().reindex(PATTERN_ORDER) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: fail/withdraw rate
ax = axes[0]
colors = [PATTERN_COLORS[p] for p in fw_rate.index]
bars = ax.barh(fw_rate.index, fw_rate.values, color=colors, edgecolor='white', height=0.6)
ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=9)
ax.set(xlabel='Failure / Withdrawal Rate (%)', title='Outcome Rate by Pattern',
       xlim=(0, 100))

# PCA scatter: first two components
ax2 = axes[1]
for pat in PATTERN_ORDER:
    mask = feats_full['pattern'] == pat
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=PATTERN_COLORS[pat], alpha=0.3, s=4, label=pat)
ax2.set(xlabel='PC1', ylabel='PC2', title='PCA — Pattern Clusters')
ax2.legend(markerscale=3, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('pattern_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFail/Withdrawal rates:')
for pat in PATTERN_ORDER:
    rate = feats_full[feats_full['pattern']==pat]['struggled'].mean() * 100
    n = (feats_full['pattern']==pat).sum()
    print(f'  {pat:22s}: {rate:5.1f}%  (n={n:,})')

In [ ]:
# ── Cross-cohort stability (2013 vs 2014) ─────────────────────────────────────
feats_2013 = feats_full[feats_full['code_presentation'].str.contains('2013')]
feats_2014 = feats_full[feats_full['code_presentation'].str.contains('2014')]

def cluster_cohort(X_sub):
    X_s = scaler.transform(X_sub)
    X_p = pca.transform(X_s)
    km = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
    return km.fit_predict(X_p)

if len(feats_2013) > 100 and len(feats_2014) > 100:
    # Cluster each cohort independently
    labels_13 = cluster_cohort(feats_2013[FEATURE_COLS].values)
    labels_14 = cluster_cohort(feats_2014[FEATURE_COLS].values)

    # Compare with main clustering labels for same students
    main_13 = feats_full.loc[feats_2013.index, 'cluster'].values
    main_14 = feats_full.loc[feats_2014.index, 'cluster'].values

    ari_13 = adjusted_rand_score(main_13, labels_13)
    ari_14 = adjusted_rand_score(main_14, labels_14)
    print(f'Cross-cohort ARI:')
    print(f'  2013 presentations: {ari_13:.3f}')
    print(f'  2014 presentations: {ari_14:.3f}')
    print(f'  → {"Strong" if min(ari_13, ari_14) > 0.60 else "Moderate"} cross-cohort reproducibility')
else:
    print('Insufficient data for cross-cohort split — using full dataset only.')

## 4 · Phase 2 — Early Classification (WP4)

In [ ]:
# ── Prepare labels from Phase 1 ───────────────────────────────────────────────
# The cluster labels derived retrospectively become the supervision signal
# for classifiers trained on early-course features only — the two-stage pipeline.

key = ['id_student','code_module','code_presentation']
pattern_labels = feats_full[key + ['pattern']].copy()

le = LabelEncoder()
le.fit(PATTERN_ORDER)

results_table = []

for cutoff, feats_cut in [(2, feats_w2), (3, feats_w3), (4, feats_w4)]:
    print(f'\n── Week {cutoff} cutoff ────────────────────────────────')

    # Merge early features with retrospective pattern labels
    merged = feats_cut.merge(pattern_labels, on=key, how='inner')
    merged = merged.dropna(subset=['pattern'])

    feat_cols = [c for c in feats_cut.columns
                 if c not in key and feats_cut[c].dtype in [np.float64, np.int64]]

    X = merged[feat_cols].fillna(0).values
    X = StandardScaler().fit_transform(X)
    y = le.transform(merged['pattern'])

    print(f'  Samples: {len(merged):,} | Features: {len(feat_cols)}')
    print(f'  Class distribution: {dict(zip(le.classes_, np.bincount(y)))}')

    # SMOTE for class imbalance
    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    X_res, y_res = sm.fit_resample(X, y)
    print(f'  After SMOTE: {len(X_res):,} samples')

    # Stratified 5-fold CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    macro_f1 = make_scorer(f1_score, average='macro', zero_division=0)

    classifiers = {
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=12, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            use_label_encoder=False, eval_metric='mlogloss',
            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
        'Logistic Regression': LogisticRegression(
            max_iter=1000, C=1.0, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1),
    }

    for name, clf in classifiers.items():
        cv_results = cross_validate(clf, X_res, y_res, cv=cv,
                                    scoring={'macro_f1': macro_f1},
                                    return_train_score=False)
        mean_f1 = cv_results['test_macro_f1'].mean()
        std_f1  = cv_results['test_macro_f1'].std()
        print(f'  {name:25s}: macro-F1 = {mean_f1:.3f} ± {std_f1:.3f}')
        results_table.append({
            'cutoff': f'Week {cutoff}',
            'classifier': name,
            'macro_f1_mean': mean_f1,
            'macro_f1_std': std_f1
        })

results_df = pd.DataFrame(results_table)
print('\n\n=== CLASSIFICATION RESULTS TABLE ===')
pivot = results_df.pivot(index='classifier', columns='cutoff', values='macro_f1_mean')
print(pivot.round(3))

In [ ]:
# ── Plot: accuracy vs earliness ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

clf_colors = {'Random Forest':'#0F766E','XGBoost':'#1D4ED8','Logistic Regression':'#B45309'}
weeks = [2, 3, 4]

for clf_name, color in clf_colors.items():
    sub = results_df[results_df['classifier'] == clf_name].sort_values('cutoff')
    f1s = sub['macro_f1_mean'].values
    stds = sub['macro_f1_std'].values
    ax.plot(weeks, f1s, 'o-', color=color, linewidth=2, label=clf_name, markersize=7)
    ax.fill_between(weeks, f1s-stds, f1s+stds, alpha=0.12, color=color)

ax.axhline(0.70, color='#B91C1C', linestyle='--', linewidth=1.5,
           label='Target threshold (0.70)')
ax.set(xlabel='Temporal cutoff (week)', ylabel='Macro-F1',
       title='Classifier Accuracy vs. Earliness — 6-Class Problem',
       xticks=[2,3,4], ylim=(0.4, 0.9))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('classifier_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP analysis — best model (RF @ Week 4) ──────────────────────────────────
print('Training final RF model at Week 4 for SHAP analysis...')

merged_w4 = feats_w4.merge(pattern_labels, on=key, how='inner').dropna(subset=['pattern'])
feat_cols_w4 = [c for c in feats_w4.columns
                if c not in key and feats_w4[c].dtype in [np.float64, np.int64]]

X_w4 = StandardScaler().fit_transform(merged_w4[feat_cols_w4].fillna(0).values)
y_w4 = le.transform(merged_w4['pattern'])

sm4 = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_res4, y_res4 = sm4.fit_resample(X_w4, y_w4)

rf_final = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf_final.fit(X_res4, y_res4)

# SHAP values (use Tree explainer — fast for RF)
sample_idx = np.random.choice(len(X_res4), min(2000, len(X_res4)), replace=False)
X_sample = X_res4[sample_idx]

explainer = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X_sample)  # list of arrays, one per class

# Global feature importance: mean |SHAP| across all classes
shap_abs_mean = np.mean([np.abs(sv).mean(0) for sv in shap_values], axis=0)
shap_importance = pd.Series(shap_abs_mean, index=feat_cols_w4).sort_values(ascending=False)

print('\nTop 15 features by mean |SHAP|:')
print(shap_importance.head(15).round(4))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
top15 = shap_importance.head(15)
ax.barh(top15.index[::-1], top15.values[::-1], color='#0F766E', edgecolor='white')
ax.set(xlabel='Mean |SHAP value|', title='Top 15 Features — Global Importance (RF @ Week 4)')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion matrix — RF @ Week 4 ───────────────────────────────────────────
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(rf_final, X_res4, y_res4,
                           cv=StratifiedKFold(5, shuffle=True,
                                              random_state=RANDOM_STATE))
cm = confusion_matrix(y_res4, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set(xlabel='Predicted', ylabel='True',
       title='Normalised Confusion Matrix — RF @ Week 4 (6-class)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-class report:')
print(classification_report(y_res4, y_pred, target_names=le.classes_, zero_division=0))

## 5 · Phase 3 — Intervention Mapping (WP5)

In [ ]:
# ── Pattern × Outcome chi-square analysis ─────────────────────────────────────
print('Pattern × Outcome association analysis')
print('='*60)

ct_full = pd.crosstab(feats_full['pattern'], feats_full['final_result'])
chi2_full, p_full, dof_full, _ = stats.chi2_contingency(ct_full)
print(f'Overall: chi2={chi2_full:.1f}, dof={dof_full}, p={p_full:.2e}')

# Per-outcome odds ratios (pattern vs all others)
print('\nPattern-level fail/withdrawal odds ratios (vs all other students):')
from scipy.stats import fisher_exact

or_results = []
for pat in PATTERN_ORDER:
    mask = feats_full['pattern'] == pat
    a = feats_full.loc[mask, 'struggled'].sum()
    b = mask.sum() - a
    c = feats_full.loc[~mask, 'struggled'].sum()
    d = (~mask).sum() - c
    table = [[a, b], [c, d]]
    or_val, p_val = fisher_exact(table)
    or_results.append({'Pattern': pat, 'Odds Ratio': or_val, 'p-value': p_val,
                        'Fail/Withdraw %': a/mask.sum()*100, 'n': mask.sum()})
    print(f'  {pat:22s}: OR={or_val:.2f}, p={p_val:.2e}, fail/withdraw={a/mask.sum()*100:.1f}%')

or_df = pd.DataFrame(or_results).set_index('Pattern')
print('\n', or_df[['Fail/Withdraw %','Odds Ratio','p-value']].round(3))

In [ ]:
# ── Intervention matrix (evidence-informed) ───────────────────────────────────
intervention_matrix = pd.DataFrame([
    {'Pattern': 'High Achiever',
     'Risk Level': 'Low',
     'Primary Intervention': 'Enrichment resources; advanced reading pathways',
     'Secondary': 'Peer mentoring or study group leadership',
     'Monitor Signal': 'Sudden engagement drop from high baseline',
     'Evidence Basis': 'Molenaar (2019); Kizilcec et al. (2013)'},
    {'Pattern': 'Steady Worker',
     'Risk Level': 'Low-Medium',
     'Primary Intervention': 'Light-touch check-in; encourage consistency',
     'Secondary': 'Extension resources if scores plateau',
     'Monitor Signal': 'Score trend — flag any sustained decline',
     'Evidence Basis': 'Rienties et al. (2016)'},
    {'Pattern': 'Procrastinator',
     'Risk Level': 'High',
     'Primary Intervention': 'Time management coaching; interim deadlines',
     'Secondary': 'Personalised study planner',
     'Monitor Signal': 'Submission lead time per week (target: >3 days)',
     'Evidence Basis': 'Tempelaar et al. (2015); Burgos et al. (2018)'},
    {'Pattern': 'Conceptual Struggler',
     'Risk Level': 'Medium',
     'Primary Intervention': 'Targeted tutoring on assessed concepts',
     'Secondary': 'Alternative learning materials; worked examples',
     'Monitor Signal': 'Score improvement rate across TMAs',
     'Evidence Basis': 'Cerezo et al. (2016); Jivet et al. (2020)'},
    {'Pattern': 'Disengaging',
     'Risk Level': 'High',
     'Primary Intervention': 'Personal tutor check-in — identify root cause',
     'Secondary': 'Academic support referral; workload review',
     'Monitor Signal': 'Weekly click count — watch for stabilisation',
     'Evidence Basis': 'Baneres et al. (2019); Rienties et al. (2016)'},
    {'Pattern': 'Ghost',
     'Risk Level': 'Critical',
     'Primary Intervention': 'Welfare outreach within 48 hours (NOT academic)',
     'Secondary': 'Check registration for withdrawal flags',
     'Monitor Signal': 'Any login activity in next 7 days',
     'Evidence Basis': 'Lonn et al. (2012); Jivet et al. (2020)'},
])

print('Intervention Recommendation Matrix')
print('='*80)
print(intervention_matrix[['Pattern','Risk Level','Primary Intervention']].to_string(index=False))

intervention_matrix.to_csv('intervention_matrix.csv', index=False)
print('\nSaved to intervention_matrix.csv')

## 6 · Phase 4 — Validation (WP6)

In [ ]:
# ── Baseline comparison: PatternDx vs binary EWS ─────────────────────────────
print('Comparison: pattern-based vs binary at-risk classification')
print('='*60)

# Binary EWS: predict struggled (Fail or Withdrawn) at week 4
binary_y = merged_w4['struggled'] if 'struggled' in merged_w4.columns else \
           feats_full.set_index(key).loc[merged_w4.set_index(key).index, 'struggled'].values

from sklearn.ensemble import RandomForestClassifier as RFC
rf_binary = RFC(n_estimators=200, max_depth=10, class_weight='balanced',
                random_state=RANDOM_STATE, n_jobs=-1)

X_bin = X_w4.copy()
y_bin = merged_w4['pattern'].map(lambda p: 1 if p in ['Ghost','Disengaging','Procrastinator'] else 0).values

cv_bin = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
bin_scores = cross_validate(rf_binary, X_bin, y_bin, cv=cv_bin,
                            scoring={'f1':'f1','precision':'precision','recall':'recall'},
                            return_train_score=False)

print('Binary EWS (at-risk vs not):')
print(f'  F1        = {bin_scores["test_f1"].mean():.3f} ± {bin_scores["test_f1"].std():.3f}')
print(f'  Precision = {bin_scores["test_precision"].mean():.3f}')
print(f'  Recall    = {bin_scores["test_recall"].mean():.3f}')

print('\nPatternDx (6-class):')
pdx_row = results_df[(results_df['classifier']=='Random Forest') &
                     (results_df['cutoff']=='Week 4')].iloc[0]
print(f'  Macro-F1  = {pdx_row["macro_f1_mean"]:.3f} ± {pdx_row["macro_f1_std"]:.3f}')
print(f'  (6-class problem — substantially harder than binary)')

print('\nAlert precision comparison:')
print('  Binary EWS: all at-risk students get identical alert')
print('  PatternDx:  each pattern maps to a distinct intervention')
print('  → PatternDx provides diagnostic specificity that binary cannot')

In [ ]:
# ── Survey results (n=2) ─────────────────────────────────────────────────────
# Raw scores from the instructor evaluation survey
print('Instructor evaluation survey results (n=2)')
print('Scale: 1=Strongly Disagree, 5=Strongly Agree')
print('='*60)

survey = pd.DataFrame([
    {'Respondent': 'R1', 'Experience': '10+ years', 'Prior EWS': 'No',
     # Scenario A (Ghost student)
     'A1_clear':1,'A1_info':2,'A1_action':1,   # Binary
     'A2_clear':2,'A2_info':3,'A2_action':2,   # PatternDx
     # Scenario B (High Achiever)
     'B1_clear':4,'B1_monitor':4,              # Binary
     'B2_clear':3,'B2_monitor':3,              # PatternDx
     'Preference':'Pattern-based',
     'Qualitative': 'B2 over-monitors high achievers. Pattern info too dense — simplify.'},
    {'Respondent': 'R2', 'Experience': '5-10 years', 'Prior EWS': 'No',
     'A1_clear':1,'A1_info':2,'A1_action':1,
     'A2_clear':3,'A2_info':4,'A2_action':3,
     'B1_clear':1,'B1_monitor':1,
     'B2_clear':4,'B2_monitor':4,
     'Preference':'Pattern-based',
     'Qualitative': ''},
])

# Actionability scores comparison
print('\nScenario A — Ghost student (most critical case):')
print('  Binary (A1)   — Clarity, Info, Action:', survey[['A1_clear','A1_info','A1_action']].values.tolist())
print('  PatternDx (A2)— Clarity, Info, Action:', survey[['A2_clear','A2_info','A2_action']].values.tolist())
print(f'  Mean binary action score:    {survey[["A1_clear","A1_info","A1_action"]].mean().mean():.2f}')
print(f'  Mean PatternDx action score: {survey[["A2_clear","A2_info","A2_action"]].mean().mean():.2f}')

print('\nScenario B — High Achiever:')
print('  Binary (B1)   — Clarity, Monitor:', survey[['B1_clear','B1_monitor']].values.tolist())
print('  PatternDx (B2)— Clarity, Monitor:', survey[['B2_clear','B2_monitor']].values.tolist())

print('\nOverall preference: both respondents chose Pattern-based')
print('\nKey qualitative finding (R1):')
print(' ', survey.loc[0, 'Qualitative'])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

labels = ['Clarity','Enough info','Would act']
x = np.arange(len(labels))
w = 0.35

for ax, (scenario, bin_cols, pdx_cols, title) in zip(axes, [
    ('A', ['A1_clear','A1_info','A1_action'], ['A2_clear','A2_info','A2_action'],
     'Scenario A — Ghost student'),
    ('B', ['B1_clear','B1_monitor'], ['B2_clear','B2_monitor'],
     'Scenario B — High Achiever')
]):
    lbl = labels[:len(bin_cols)]
    xp = np.arange(len(lbl))
    bin_means = survey[bin_cols].mean().values
    pdx_means = survey[pdx_cols].mean().values
    bars1 = ax.bar(xp - w/2, bin_means, w, label='Binary EWS', color='#94A3B8', alpha=0.85)
    bars2 = ax.bar(xp + w/2, pdx_means, w, label='PatternDx', color='#0F766E', alpha=0.85)
    ax.set(xticks=xp, xticklabels=lbl, ylim=(0, 5.5),
           ylabel='Rating (1–5)', title=title)
    ax.axhline(3, color='gray', linestyle=':', alpha=0.5)
    ax.legend(fontsize=8)
    ax.bar_label(bars1, fmt='%.1f', fontsize=8, padding=2)
    ax.bar_label(bars2, fmt='%.1f', fontsize=8, padding=2)

plt.suptitle('Instructor Evaluation: Binary EWS vs PatternDx (n=2)', y=1.03)
plt.tight_layout()
plt.savefig('survey_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Summary — Dissertation Numbers

In [ ]:
# ── All key numbers for dissertation ─────────────────────────────────────────
print('='*70)
print('DISSERTATION SUMMARY — KEY NUMBERS')
print('='*70)

print(f'\nDataset: {len(info):,} students, {info["code_module"].nunique()} courses')
print(f'         {len(stvle):,} VLE interactions')

print('\nFeature Engineering:')
print(f'  {len(FEATURE_COLS)} features across 5 categories')
print(f'  Computed at weeks 2, 3, and 4')

print(f'\nPCA: {X_scaled.shape[1]} features → {X_pca.shape[1]} components (95% variance)')

print(f'\nPattern Discovery (WP3):')
print(f'  Optimal k = {OPTIMAL_K} (converged across K-means, HDBSCAN, GMM)')
print(f'  Silhouette score (K-means): {sil_km:.3f}')
print(f'  Chi-square: χ²={chi2:.1f}, p={p:.2e}')
print(f'  ARI (K-means vs HDBSCAN): {ari_km_hdb:.3f}')
print(f'  ARI (K-means vs GMM): {ari_km_gmm:.3f}')

print(f'\nClassification results (Macro-F1):')
for _, row in results_df[results_df['classifier']=='Random Forest'].iterrows():
    print(f'  RF @ {row["cutoff"]}: {row["macro_f1_mean"]:.3f} ± {row["macro_f1_std"]:.3f}')

print(f'\nSurvey (n=2):')
print(f'  Both respondents preferred PatternDx over binary EWS')
print(f'  Binary Ghost actionability mean: 1.33/5')
print(f'  PatternDx Ghost actionability mean: 2.67/5')
print(f'  Key limitation: R1 noted High Achiever output is over-specified')
print('='*70)